# Ray 2.60: A Practical Introduction to LLM Post-Training

This notebook introduces supervised fine-tuning (SFT) with Ray Train, including distributed execution, data handling, model checkpoints, and the limits of recovery. It also explains how to extend the workflow to LoRA/QLoRA, DPO, and GRPO.

**Environment:** Linux with an NVIDIA GPU. A single GPU with 24 GB or more of VRAM is a conservative starting point, not a measured minimum. Multi-GPU training requires NCCL. The example uses `Qwen/Qwen2.5-0.5B-Instruct`; check model access, licensing, and memory requirements. Use Linux or WSL2 for GPU training rather than native Windows.

> Version check: November 17, 2026. Ray's official documentation and PyPI reported version 2.60.0. Recheck the official API when upgrading.

## 1. Post-Training and the Ray Components

SFT optimizes next-token loss using instruction/answer examples. DPO directly optimizes a policy from preference pairs. GRPO/PPO also require rollouts and rewards from a model or a rule-based function. Ray provides distributed scheduling: Ray Data handles data pipelines, Ray Train coordinates training workers, Ray Tune supports hyperparameter search, and Ray Serve can host online inference.

Start with the inspectable SFT workflow in this notebook. Larger preference-optimization systems often use frameworks such as verl, TRL, or OpenRLHF, with Ray coordinating applicable data, training, or rollout workloads.

In [ ]:
# First install a CUDA-compatible PyTorch build: https://pytorch.org/get-started/locally/
# These selected dependency versions have not been GPU integration-tested here; review upgrades carefully.
%pip install "ray[train,data,default]==2.60.0" "transformers==4.57.1" "datasets==4.2.0" "accelerate==1.10.1" "peft==0.17.1" safetensors
# Restart the kernel after installation; record the successful environment with pip freeze.

In [ ]:
import os
import ray
import torch

print("Ray:", ray.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Start Ray and Inspect Resources

Use `ray.init()` for local experiments. For multi-node training, connect to an existing Ray cluster with matching Ray versions, Python environments, and dependencies across the head and workers. Runtime environments can install worker dependencies; larger deployments commonly use a consistent container image.

In [ ]:
if not ray.is_initialized():
    ray.init()  # For an existing cluster, use ray.init(address="auto")

print(ray.cluster_resources())
print(ray.available_resources())

## 2.1 Learn Tasks, ObjectRefs, and Actors on CPU

A task handles independent computation, such as scoring an answer with a reward rule. An actor is a stateful process, such as a rollout model kept resident on a GPU.
`.remote()` returns an ObjectRef immediately; `ray.get()` waits for the result. Avoid calling `get` after each submission inside a loop, as that serializes the waiting.
The following example computes toy rewards. It does not call an LLM or implement RL training.

In [ ]:
@ray.remote(num_cpus=1)
def rule_reward(answer, expected):
    return float(answer.strip() == expected)

refs = [rule_reward.remote(answer, "4") for answer in ["4", "5", " 4 "]]
print("parallel rewards:", ray.get(refs))

@ray.remote(num_cpus=1)
class RewardCounter:
    def __init__(self):
        self.count = 0
    def record(self, rewards):
        self.count += len(rewards)
        return {"samples": self.count, "mean_reward": sum(rewards) / len(rewards)}

counter = RewardCounter.remote()
print(ray.get(counter.record.remote(ray.get(refs))))
ray.kill(counter)  # Release the actor's CPU resources

**Resource requests are not memory limits.** `num_gpus=1` is a scheduling declaration; it does not cap VRAM. Fractional GPU allocations do not isolate GPU memory either.
Load a GPU actor's model once in `__init__` and reuse it for inference. Training and rollout processes that occupy all GPUs can leave other workers waiting or cause resource deadlocks.
At scale, use `ray.wait()` to bound outstanding ObjectRefs instead of accumulating every generated result in the object store.

## 3. Build a Small Instruction Dataset

Use a few synthetic examples to check the code path, not to measure model quality. For production, use authorized, deduplicated, cleaned data with sensitive information handled appropriately. Separate training and validation examples, and inspect formatting, length distributions, duplication, and private information. The chat template must match the model.

In [ ]:
from datasets import Dataset

# Split by distinct question before repeating training samples to prevent validation leakage.
examples = [
    {"instruction": "Explain Ray in one sentence.", "response": "Ray is a framework for distributed Python computing."},
    {"instruction": "What is SFT?", "response": "SFT continues model training using examples with target answers."},
    {"instruction": "How do you reverse a list in Python?", "response": "You can use items[::-1]."},
    {"instruction": "What is a checkpoint used for?", "response": "It saves model and training state for evaluation or recovery."},
]
train_ds = {
    "train": Dataset.from_list(examples[:3] * 32),
    "test": Dataset.from_list(examples[3:]),
}
train_ds

## 4. Preprocess with Ray Data

Ray Data parallelizes file reading, filtering, transformation, and partitioning. Here, we convert Hugging Face Datasets into Ray Datasets and apply the model's chat template. Tokenization happens inside each training worker. At scale, decide whether to tokenize ahead of time based on throughput, CPU/GPU utilization, and caching requirements.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
# Pin a model commit hash for reproducible experiments instead of the moving main branch.
MODEL_REVISION = "main"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_chat_text(row):
    messages = [
        {"role": "user", "content": row["instruction"]},
        {"role": "assistant", "content": row["response"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

ray_train_ds = ray.data.from_huggingface(train_ds["train"]).map(to_chat_text)
ray_valid_ds = ray.data.from_huggingface(train_ds["test"]).map(to_chat_text)
ray_train_ds.take(2)

## 5. Ray Train + LoRA SFT

LoRA is enabled by default: only adapters on selected attention projections are trained. Set `USE_LORA=False` for full fine-tuning.
Ray starts one process per GPU. Transformers/Accelerate establishes DDP, and Trainer applies distributed sampling to the same HF Dataset on each worker.
**Do not also shard this dataset manually with Ray: that would shard it twice.**

This small example uses `take_all()` to collect processed rows. Do not copy this approach for large datasets. Instead, use Ray Train `datasets` / `get_dataset_shard()` with an appropriate custom DataLoader, or let every worker read the same disk-backed HF Dataset and use Trainer's distributed sampling.

For simplicity, the loss covers the entire chat sequence, including user tokens. Instruction SFT often supervises only assistant tokens. Generate label masks using the model's template and inspect them sample by sample; do not infer token boundaries from character positions alone.

Effective global batch size = per-device batch size × gradient accumulation steps × worker count. With DDP, each GPU still holds the complete base model. LoRA does not shard model weights.

In [ ]:
import tempfile
from pathlib import Path
from ray import train
from ray.train import Checkpoint, RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer
from ray.train.huggingface.transformers import prepare_trainer

# Collect rows on the driver only for this tiny demonstration.
train_rows = ray_train_ds.take_all()
eval_rows = ray_valid_ds.take_all()
MAX_LENGTH = 256
assert train_rows and eval_rows
print("train/eval rows:", len(train_rows), len(eval_rows))

In [ ]:
def train_loop_per_worker(config):
    import torch
    from datasets import Dataset
    from transformers import (
        AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments,
        DataCollatorForSeq2Seq, set_seed,
    )
    from peft import LoraConfig, get_peft_model

    set_seed(42)
    context = train.get_context()
    tok = AutoTokenizer.from_pretrained(config["model_id"], revision=config["revision"])
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    model = AutoModelForCausalLM.from_pretrained(
        config["model_id"], revision=config["revision"],
        torch_dtype=torch.bfloat16 if use_bf16 else torch.float32,
    )
    model.config.use_cache = False
    if config["use_lora"]:
        model = get_peft_model(model, LoraConfig(
            task_type="CAUSAL_LM", r=8, lora_alpha=16, lora_dropout=0.05,
            target_modules=["q_proj", "v_proj"],
        ))
        model.print_trainable_parameters()

    def tokenize(batch):
        encoded = tok(batch["text"], truncation=True,
                      max_length=config["max_length"], add_special_tokens=False)
        encoded["labels"] = [ids.copy() for ids in encoded["input_ids"]]
        return encoded

    def make_dataset(rows):
        return Dataset.from_list(rows).map(tokenize, batched=True, remove_columns=["text"])

    # Mask padding labels with -100 while preserving supervision on real EOS tokens.
    collator = DataCollatorForSeq2Seq(tok, label_pad_token_id=-100, pad_to_multiple_of=8)
    with tempfile.TemporaryDirectory(prefix="ray-sft-") as workdir:
        args = TrainingArguments(
            output_dir=workdir, max_steps=config["max_steps"],
            per_device_train_batch_size=config["batch_size"],
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=config["grad_accum"],
            learning_rate=config["learning_rate"],
            logging_steps=1, save_strategy="no", eval_strategy="no",
            bf16=use_bf16, fp16=False,
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            ddp_find_unused_parameters=False, report_to=[], seed=42,
        )
        hf_trainer = Trainer(
            model=model, args=args, data_collator=collator,
            train_dataset=make_dataset(config["train_rows"]),
            eval_dataset=make_dataset(config["eval_rows"]),
        )
        hf_trainer = prepare_trainer(hf_trainer)
        output = hf_trainer.train()
        metrics = hf_trainer.evaluate()
        metrics["train_loss"] = output.training_loss
        checkpoint = None
        if context.get_world_rank() == 0:
            export = Path(workdir) / "export"
            hf_trainer.save_model(str(export))
            tok.save_pretrained(str(export))
            checkpoint = Checkpoint.from_directory(str(export))
        # Every rank calls report equally often; only rank 0 uploads model files.
        train.report(metrics, checkpoint=checkpoint)

## 6. Launch Training

This smoke run performs 10 optimizer steps to check the workflow. It does not establish an improvement in model quality.
The first run downloads the model. Set `RUN_TRAINING=True` below to launch training.

The default LoRA learning rate is 2e-4; choose a separate rate for full fine-tuning, for example starting experiments at 2e-5.
A local `storage_path` is suitable only for a single node. Multi-node runs need persistent storage accessible from all nodes.
Automatic retries are not enabled here, so a persistent GPU OOM does not repeatedly restart an unsuccessful job.

In [ ]:
RUN_TRAINING = False
NUM_WORKERS = 1
USE_LORA = True
result = None
if RUN_TRAINING:
    assert torch.cuda.is_available(), "Run in a CUDA-enabled Linux or WSL2 environment."
    assert ray.cluster_resources().get("GPU", 0) >= NUM_WORKERS
    trainer = TorchTrainer(
        train_loop_per_worker,
        train_loop_config={
            "model_id": MODEL_ID, "revision": MODEL_REVISION,
            "max_steps": 10, "max_length": MAX_LENGTH,
            "batch_size": 1, "grad_accum": 4,
            "learning_rate": 2e-4 if USE_LORA else 2e-5,
            "use_lora": USE_LORA,
            "train_rows": train_rows, "eval_rows": eval_rows,
        },
        scaling_config=ScalingConfig(
            num_workers=NUM_WORKERS, use_gpu=True,
            resources_per_worker={"CPU": 1, "GPU": 1},
        ),
        run_config=RunConfig(
            name="ray-llm-sft", storage_path=str(Path("ray_results").resolve()),
        ),
    )
    result = trainer.fit()
    print(result.metrics)
    print(result.checkpoint)
else:
    print("GPU training skipped. Set RUN_TRAINING=True when the environment is ready.")

## 7. Load the Adapter or Full Model for Generation

A LoRA checkpoint contains the adapter and tokenizer; inference also needs the original base model. This example generates on CPU, which can be slow. Move the model and inputs to an available CUDA device for faster inference.
The validation set contains only one independent question and is not a credible benchmark. For real evaluation, fix the task set, model revision, and decoding parameters, then compare the base and fine-tuned models for correctness, format compliance, and safety.

**Recovery boundary:** This example exports model/adapter weights without optimizer, scheduler, or RNG state. It cannot resume training exactly. Exact recovery requires a complete HF Trainer checkpoint, transported through a Ray checkpoint and loaded inside each worker using `trainer.train(resume_from_checkpoint=...)`. All ranks must resume from the same training step.

In [ ]:
if result is not None:
    from transformers import AutoModelForCausalLM
    from peft import PeftModel
    with result.checkpoint.as_directory() as checkpoint_dir:
        tok = AutoTokenizer.from_pretrained(checkpoint_dir)
        if USE_LORA:
            base = AutoModelForCausalLM.from_pretrained(
                MODEL_ID, revision=MODEL_REVISION, torch_dtype=torch.float32)
            model = PeftModel.from_pretrained(base, checkpoint_dir)
        else:
            model = AutoModelForCausalLM.from_pretrained(
                checkpoint_dir, torch_dtype=torch.float32)
        model.eval()
        messages = [{"role": "user", "content": "What is a checkpoint used for?"}]
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt", add_special_tokens=False)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=64, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
else:
    print("Run this cell again after training.")

## 8. Scale from One Machine to a Cluster

- Start Ray head and worker nodes. Confirm that `ray status` reports the expected GPUs, then connect with `ray.init(address="auto")`.
- `ScalingConfig(num_workers=N, use_gpu=True)` sets the number of training processes. DDP, FSDP, and DeepSpeed differ in memory partitioning and communication. Models that exceed per-device memory may require FSDP or DeepSpeed.
- Point `RunConfig(storage_path=...)` to persistent storage accessible to all nodes. Verify recovery behavior, checkpoint completeness, and dependency consistency.
- Use Ray Data to read partitioned Parquet/JSONL files, filter invalid examples, manage sequence lengths, and monitor throughput. Avoid loading the entire dataset into the notebook driver.
- Measure tokens/s, GPU utilization, data wait time, and communication overhead. More GPUs do not guarantee linear speedups.

## 9. Next Steps: LoRA, DPO, and GRPO

**LoRA/QLoRA:** PEFT injects adapters after model loading and trains only adapter parameters, reducing optimizer-state and gradient memory. QLoRA also needs a compatible bitsandbytes environment for 4-bit weights. Verify CUDA, bitsandbytes, and PEFT compatibility on every worker.

**DPO:** Prepare `(prompt, chosen, rejected)` preference pairs and use TRL's `DPOTrainer` or another suitable training loop. Ray Train manages workers and resources. Pay attention to reference-model memory, sequence length, and preference quality.

**GRPO/PPO:** Schedule rollout inference and reward computation alongside policy training. Resource orchestration, asynchronous sampling, and checkpoint behavior become more complex. A dedicated framework such as verl provides a more suitable starting point; follow its own Ray deployment instructions.

Production workflows also need versioned data, recoverable persistent checkpoints, experiment tracking, offline evaluations, model cards and license checks, safety evaluations, and training/inference format compatibility checks.

## Official References

- [Ray Train](https://docs.ray.io/en/latest/train/train.html)
- [Ray Train + Hugging Face Transformers](https://docs.ray.io/en/latest/train/api/doc/ray.train.huggingface.transformers.prepare_trainer.html)
- [Ray Data](https://docs.ray.io/en/latest/data/data.html)
- [Ray examples](https://docs.ray.io/en/latest/ray-overview/examples.html)
- [Ray on GitHub](https://github.com/ray-project/ray)

This notebook demonstrates basic Ray training orchestration. Ray APIs, Transformers Trainer arguments, and integration support can change across versions. Consult the corresponding documentation before upgrading.

## 10. DPO and GRPO: Data and Resource Requirements

| Method | Data | Main model roles | First checks |
|---|---|---|---|
| SFT | Prompt + answer | Policy | Assistant label mask, template |
| DPO | Prompt + chosen + rejected | Policy, reference | Pair quality, policy/reference initialization |
| GRPO | Prompt, verifiable answer or reward function | Policy, rollout, optional reference depending on configuration | Within-group reward variation, KL, length bias |

DPO optimizes relative log probabilities of preferred and rejected answers to the same prompt; it does not require online rollouts.
GRPO samples a group of answers per prompt, constructs advantages from relative rewards, and updates the policy.
When every reward in a group is identical, normalization provides no useful relative reward signal. Track within-group reward standard deviation, KL, output length, and external evaluation results in addition to mean reward.

Ray orchestrates processes and resources; it does not automatically implement the correct DPO/GRPO loss or weight synchronization.
TRL is a useful starting point for single-machine algorithm experiments. verl and OpenRLHF have their own Ray orchestration and version requirements. Use their prescribed environments rather than assuming compatibility with this notebook's dependencies.

In [ ]:
# Minimal preference-data structure; this does not launch training.
preference_pair = {
    "prompt": [{"role": "user", "content": "What is 2+2?"}],
    "chosen": [{"role": "assistant", "content": "4"}],
    "rejected": [{"role": "assistant", "content": "5"}],
}
assert preference_pair["chosen"] != preference_pair["rejected"]

# Illustrative GRPO advantages; real training also needs policy loss, KL, clipping, and more.
rewards = torch.tensor([1., 0., 1., 0.])
advantages = (rewards - rewards.mean()) / (rewards.std(unbiased=False) + 1e-6)
print("group advantages:", advantages.tolist())

## 11. Troubleshooting and Exercises

| Symptom | What to inspect |
|---|---|
| Workers remain PENDING | `ray.available_resources()`; actors holding CPU/GPU resources |
| CUDA OOM | Reduce length/batch size; use LoRA; DDP does not shard base weights |
| Multi-node NCCL hangs | Network connectivity, CUDA/drivers, matching communication steps across ranks |
| NaN loss | Labels, learning rate, precision, hardware support for bf16 |
| Checkpoint cannot resume training | Adapter export versus complete optimizer/RNG training state |
| Driver slows down as data grows | Replace `take_all()` with partitioned storage and streaming |

Exercises:

1. Use two GPUs and calculate the effective global batch size.
2. Compare throughput while keeping global batch size fixed.
3. Print tokens and label masks for three samples.
4. Replace the toy validation data with an independent evaluation set.
5. Add complete training checkpoints, interrupt a run, and verify the resumed step.

GPU training has not been executed for this notebook. No measured loss, throughput, or quality improvement is claimed. Syntax validation does not replace an integration test on the target hardware.

### Additional Official Links

- [Ray 2.60.0 release source](https://github.com/ray-project/ray/tree/ray-2.60.0)
- [Ray Core concepts](https://docs.ray.io/en/latest/ray-core/key-concepts.html)
- [Ray Hugging Face integration source](https://github.com/ray-project/ray/blob/ray-2.60.0/python/ray/train/huggingface/transformers/_transformers_utils.py)
- [TRL DPO](https://huggingface.co/docs/trl/dpo_trainer)
- [TRL GRPO](https://huggingface.co/docs/trl/grpo_trainer)
- [verl](https://github.com/volcengine/verl)
- [OpenRLHF](https://github.com/OpenRLHF/OpenRLHF)

"Latest" refers to the Ray release checked on the stated date. This tutorial does not claim to use the newest model or the newest version of every dependency.

In [ ]:
# Run manually after local experiments to shut down this process's Ray runtime:
# ray.shutdown()